In [1]:
import glob
import os
import sys
import xml.etree.ElementTree as ET
import json
import time
import pytz
import datetime as dt
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import ast
import netCDF4 as nc
import xarray as xr
import re
from datetime import datetime
from geopy.distance import geodesic
from netCDF4 import Dataset

import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
from scipy import stats
from scipy.stats import linregress
from functools import reduce
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from scipy.stats import zscore
from dominance_analysis import Dominance

from joblib import Parallel, delayed
import multiprocessing

# Define the haversine_distance function to calculate distances in meters
def haversine_distance(lon1, lat1, lon2, lat2):
    # Convert decimal degrees to radians
    lon1_rad, lat1_rad = np.radians(lon1), np.radians(lat1)
    lon2_rad, lat2_rad = np.radians(lon2), np.radians(lat2)

    # Haversine formula
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    R = 6371000  # Earth's radius in meters
    return c * R  # Distance in meters


# Function to process each variable
def process_variable(ds_var, mask_da_expanded):
    # Select data after 1970
    # Apply the mask, expanding it to match the dataset dimensions
    mask_expanded, _ = xr.broadcast(mask_da_expanded, ds_var)
    masked_ds = ds_var.where(mask_expanded)
    
    # Resample to yearly mean over time
    yearly_mean = masked_ds.groupby('time.year').mean(dim=['time', 'lat', 'lon'])
    
    # Convert to DataFrame
    df_yearly_mean = yearly_mean.to_dataframe().reset_index()
    
    return df_yearly_mean

def process_variable_no_mask(ds_var):
    yearly_mean = ds_var.groupby('time.year').mean(dim=['time', 'lat', 'lon'])
    df_yearly_mean = yearly_mean.to_dataframe().reset_index()
    
    return df_yearly_mean


def load_and_preprocess_dataset(file_path):
    try:
        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"File {file_path} not found.")
        
        ds_var = xr.open_dataset(file_path, decode_times=False)
        
        # Coordinate renaming logic
        coord_rename_dict = {}
        lon_names = ['longitude', 'Longitude', 'LONGITUDE', 'X', 'x', 'lon_FULL']
        lat_names = ['latitude', 'Latitude', 'LATITUDE', 'Y', 'y', 'lat_FULL']
        time_names = ['time_counter', 'time', 'Time', 'TIME']

        for name in lon_names:
            if name in ds_var.coords or name in ds_var.variables:
                coord_rename_dict[name] = 'lon'
                break

        for name in lat_names:
            if name in ds_var.coords or name in ds_var.variables:
                coord_rename_dict[name] = 'lat'
                break

        for name in time_names:
            if name in ds_var.coords or name in ds_var.variables:
                coord_rename_dict[name] = 'time'
                break

        if coord_rename_dict:
            ds_var = ds_var.rename(coord_rename_dict)

        time_var      = ds_var['time']
        time_units    = time_var.attrs.get('units', '')
        time_calendar = time_var.attrs.get('calendar', 'standard')
        time_values   = time_var.values

       # If the units string itself is "months since …", force the 360_day calendar:
        if time_units.startswith('months since'):
            calendar = '360_day'
            units    = time_units
        # If it's "years since …" on a noleap/365_day calendar, switch to common_years:
        elif time_calendar in ('noleap', '365_day') and time_units.startswith('years since'):
            calendar = time_calendar
            units    = time_units.replace('years since', 'common_years since')
        else:
            calendar = time_calendar
            units    = time_units

        dates = nc.num2date(time_values, units=units, calendar=calendar)
        ds_var['time'] = dates
        
        return ds_var

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None



def calculate_yearly_means(var_name, ds_var):
    try:        

        var = ds_var[var_name]
        lat = ds_var['lat']
        lon = ds_var['lon']

        # Earth's radius in meters
        R = 6371000

        latitudes = lat.values
        longitudes = lon.values
        
        #Check if latitude is in the order from North to South
        if latitudes[0] > latitudes[-1]:
            latitudes = latitudes[::-1]
            var = var.sel(lat=latitudes)
        
        #Calculate pixel resolution
        dlat = np.abs(np.diff(latitudes).mean())
        dlon = np.abs(np.diff(longitudes).mean())

        lat_rad = np.deg2rad(latitudes)
        dlat_rad = np.deg2rad(dlat)
        dlon_rad = np.deg2rad(dlon)

        lat_bounds = np.concatenate((
            [latitudes[0] - dlat / 2],
            (latitudes[:-1] + latitudes[1:]) / 2,
            [latitudes[-1] + dlat / 2]
        ))
        lat_bounds_rad = np.deg2rad(lat_bounds)

        area = R**2 * dlon_rad * (np.sin(lat_bounds_rad[1:]) - np.sin(lat_bounds_rad[:-1]))
        area = np.tile(area[:, np.newaxis], (1, len(longitudes)))

        area_da = xr.DataArray(
            area,
            dims=['lat', 'lon'],
            coords={'lat': latitudes, 'lon': longitudes}
        )

        var_area_weighted = var * area_da 
        #* 365 * 24 * 60 * 60  # Convert to annual values
        
        #time_coord = 'common_year' if 'common_years' in var_area_weighted.coords else 'year'
        #annual_mean = var_area_weighted.groupby(f'time.{time_coord}').mean(dim='time')

        annual_mean = var_area_weighted.groupby('time.year').mean(dim='time')
        annual_sum = annual_mean.sum(dim=['lat', 'lon']) 
        #* 1e-12  # Convert to petagrams
        
        # Ensure the DataArray has a name before converting to DataFrame
        if not annual_sum.name:
            annual_sum.name = var_name

        return annual_sum.to_dataframe().reset_index()

    except Exception as e:
        print(f"Error calculating yearly means for variable '{var_name}': {e}")
        return None

def calculate_yearly_means_each_pixel(var_name, ds_var):
    try:        

        var = ds_var[var_name]
        lat = ds_var['lat']
        lon = ds_var['lon']

        # Earth's radius in meters
        R = 6371000

        latitudes = lat.values
        longitudes = lon.values
        
        #Check if latitude is in the order from North to South
        if latitudes[0] > latitudes[-1]:
            latitudes = latitudes[::-1]
            var = var.sel(lat=latitudes)
        
        #Calculate pixel resolution
        dlat = np.abs(np.diff(latitudes).mean())
        dlon = np.abs(np.diff(longitudes).mean())

        lat_rad = np.deg2rad(latitudes)
        dlat_rad = np.deg2rad(dlat)
        dlon_rad = np.deg2rad(dlon)

        lat_bounds = np.concatenate((
            [latitudes[0] - dlat / 2],
            (latitudes[:-1] + latitudes[1:]) / 2,
            [latitudes[-1] + dlat / 2]
        ))
        lat_bounds_rad = np.deg2rad(lat_bounds)

        area = R**2 * dlon_rad * (np.sin(lat_bounds_rad[1:]) - np.sin(lat_bounds_rad[:-1]))
        area = np.tile(area[:, np.newaxis], (1, len(longitudes)))

        area_da = xr.DataArray(
            area,
            dims=['lat', 'lon'],
            coords={'lat': latitudes, 'lon': longitudes}
        )

        var_area_weighted = var * area_da 
        #* 365 * 24 * 60 * 60  # Convert to annual values
        
        #time_coord = 'common_year' if 'common_years' in var_area_weighted.coords else 'year'
        #annual_mean = var_area_weighted.groupby(f'time.{time_coord}').mean(dim='time')

        annual_mean = var_area_weighted.groupby('time.year').mean(dim='time')
        #* 1e-12  # Convert to petagrams
        
        if not annual_mean.name:
            annual_mean.name = var_name

        return annual_mean.to_dataframe().reset_index()

    except Exception as e:
        print(f"Error calculating yearly means for variable '{var_name}': {e}")
        return None

    
#COMBINE 2 Masks
def mask_df(var_name, ds_var, mask):
    
    ds2_masked = ds_var.where(mask, drop=False)
    ds_pft0 = ds2_masked.isel(PFT=0)  
    ds_pft1 = ds2_masked.isel(PFT=1) 

    if var_name in ds_pft0.variables and var_name in ds_pft1.variables:
        data_pft0 = ds_pft0[var_name]
        data_pft1 = ds_pft1[var_name]
    else:
        raise ValueError(f"Variable '{var_name}' not found in one or both PFT datasets.")
    
    #Combine 2 masks
    combined_data = data_pft0.where(~data_pft0.isnull(), 0) + data_pft1.where(~data_pft1.isnull(), 0)
    combined_data = combined_data.where(combined_data != 0)
    
    final = ds_var.where(~combined_data, drop=False)

    return final

Loading BokehJS ...

In [2]:
os.chdir('/Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/data/TRENDYv10/downloads/')

# Site weight yearly mean

In [6]:
model_list = ["CABLE-POP", 'ISBA-CTRIP', 'LPX-Bern', 'ORCHIDEE', 'ORCHIDEEv3', 'CLM5.0', 'CLASSIC', 'CLASSIC-N']

scenario = ['S0', 'S1', 'S2']
var_names = ['cLeaf', 'cRoot', 'cWood', 'gpp', 'ra']
start_time = time.time()

for i in model_list:
    # Define parameters
    model_name = i
    # Load site information
    flux_site_info = pd.read_csv("/Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/data/cabon/Cabonetal_site_info.csv")
    flux_site_info = flux_site_info[flux_site_info["On-site RW"] == True]
    
    lat_points = np.array(flux_site_info["Lat."])
    lon_points = np.array(flux_site_info["Lon."])

    # Template file for grid info
    file_path_template = f"{model_name}/{model_name}_{scenario[2]}_gpp.nc"
    df_template = load_and_preprocess_dataset(file_path_template)

    #convert lat and lon if the model use range lon (0, 360) rather than (-180, 180)
    #df_template = df_template.assign_coords(
    #    lon = df_template['lon'].where(df_template['lon'] < 180, df_template['lon'] - 360)
    #)
        
    lat = df_template['lat'].values
    lon = df_template['lon'].values
    lat_indices = [np.abs(lat - lat_p).argmin() for lat_p in lat_points]
    lon_indices = [np.abs(lon - lon_p).argmin() for lon_p in lon_points]
    
    
    variable_shape = (len(lat), len(lon))
    mask = np.zeros(variable_shape, dtype=bool)

    for i, j in zip(lat_indices, lon_indices):
        mask[i, j] = True

    mask_da = xr.DataArray(mask, coords={'lat': lat, 'lon': lon}, dims=['lat', 'lon'])
    
    # Output directory
    output_dir = f"/Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/results/TRENDYv10/31_site_weighted/{model_name}"
    os.makedirs(output_dir, exist_ok=True)

    # Function to process each variable
    def process_and_save_variable(var_name):
        for j in scenario:

            file_path = f"{model_name}/{model_name}_{j}_{var_name}.nc"
            ds_var = load_and_preprocess_dataset(file_path)
            
            #convert lat and lon if neccesary
            ds_var = ds_var.assign_coords(lon = ds_var['lon'].where(ds_var['lon'] < 180, ds_var['lon'] - 360))
        
            if ds_var is not None:
                df_with_mask = ds_var.where(mask_da)
                df = calculate_yearly_means_each_pixel(var_name, df_with_mask)

                if df is not None:
                    output_file = os.path.join(output_dir, f"{var_name}_{j}_31_site_weighted_yearly_mean.csv")
                    df.to_csv(output_file, index=False)
                    print(f"Saved {var_name} 31 site weighted yearly mean to {output_file}")

    # Get number of CPU cores to use (all except one)
    num_cores = multiprocessing.cpu_count() - 1

    # Parallel execution
    Parallel(n_jobs=num_cores)(delayed(process_and_save_variable)(var) for var in var_names)

    end_time = time.time()

    print(f"Execution time: {end_time - start_time:.2f} seconds")

Saved cRoot 31 site weighted yearly mean to /Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/results/TRENDYv10/31_site_weighted/CLM5.0/cRoot_S0_31_site_weighted_yearly_mean.csv
Saved cRoot 31 site weighted yearly mean to /Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/results/TRENDYv10/31_site_weighted/CLM5.0/cRoot_S1_31_site_weighted_yearly_mean.csv
Saved cRoot 31 site weighted yearly mean to /Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/results/TRENDYv10/31_site_weighted/CLM5.0/cRoot_S2_31_site_weighted_yearly_mean.csv
Saved ra 31 site weighted yearly mean to /Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/results/TRENDYv10/31_site_weighted/CLM5.0/ra_S0_31_site_weighted_yearly_mean.csv
Saved ra 31 site weighted yearly mean to /Users/ngocnguyen/Library/CloudStorage/Box-Box/Projects/biomass_paper/results/TRENDYv10/31_site_weighted/CLM5.0/ra_S1_31_site_weighted_yearly_mean.csv
Saved ra 31 site weigh